# Lab 1 at home · Two tables of your own in DuckDB

**Due:** Sunday, October 25 at 11:59 PM · **Submit:** your repo URL on CourseWorks

In class you loaded one table (penguins), found a problem and fixed it. Now do the same with **two related tables you choose**. In Session 2 we'll join them, so pick two that can be connected.

You will:
1. Pick two related public datasets and upload them
2. Load both into DuckDB and check what DuckDB decided about each column
3. Find the column the two tables share (the **key**) and check how well they match
4. Write five SQL queries that help explain your data
5. Write observations and a dataset card for each table

## 1. Pick two related tables

**Related** means they share a column you could match rows on: an ID, a name, a ZIP code, a country code, a date.

**Example pairs**
- **World Bank:** any indicator download (e.g. GDP per country per year) comes with a second file of country details (region, income group). The shared column is the country code.
- **Our World in Data:** two charts for the same countries, e.g. life expectancy and health spending. Shared columns: country and year.
- **NYC Open Data:** 311 complaints and population by ZIP code, or the street tree census and neighborhood boundaries. Shared column: ZIP code or neighborhood.
- **Your own idea:** any two public tables that share a column.

**Rules**
- **Public, open data only.** Your repo is public, and later this data goes to a free AI tier. Nothing from work, nothing personal, nothing behind a login.
- **CSV format.** If a file is Excel, open it and save it as CSV first.
- **Each file under 100 MB.** Between 1,000 and 1,000,000 rows is a good size for the main table; the second can be small.
- **Reuse allowed.** Check each license or terms page, and note it in your dataset card.

**Places to look:** [NYC Open Data](https://opendata.cityofnewyork.us) · [data.gov](https://data.gov) · [World Bank Open Data](https://data.worldbank.org) · [Our World in Data](https://ourworldindata.org) · [Kaggle Datasets](https://www.kaggle.com/datasets) (check each license)

## 2. Upload them

1. Download both CSVs to your computer.
2. In this Codespace, drag both files into the **`data`** folder in the file list on the left.
3. Rename each to something short with no spaces, e.g. `gdp.csv` and `countries.csv` (right-click → **Rename**).
4. Fill in the cell below and run it. Table names should be short, lowercase, no spaces: you'll type them in your SQL.

In [1]:
FILE_A  = "crashes.csv"     # <- your main table's file name
TABLE_A = "crashes"                   # <- a short name for it, e.g. "gdp"

FILE_B  = "speeding.csv"   # <- the related table's file name
TABLE_B = "speeding"                 # <- a short name for it, e.g. "countries"

In [2]:
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_A = ROOT / "data" / FILE_A
DATA_B = ROOT / "data" / FILE_B

missing = [p.name for p in (DATA_A, DATA_B) if not p.exists()]
if missing:
    print("Can't find:", ", ".join(missing))
    print("Files in data/ right now:")
    for f in sorted((ROOT / "data").iterdir()):
        print("  ", f.name)
else:
    con = duckdb.connect(str(ROOT / "lab1_home.duckdb"))
    for p in (DATA_A, DATA_B):
        print(f"Found data/{p.name} ({p.stat().st_size / 1_000_000:.1f} MB)")
    print("Connected.")

Found data/crashes.csv (67.4 MB)
Found data/speeding.csv (1.7 MB)
Connected.


## 3. First look

Preview the first 10 rows of each raw file before loading.

In [3]:
con.sql(f"SELECT * FROM read_csv('{DATA_A}') LIMIT 10")

┌────────────┬────────────┬──────────┬──────────┬───────────┬───────────┬────────────────────────────┬──────────────────────┬─────────────────────┬───────────────────────────┬───────────────────────────┬──────────────────────────┬───────────────────────────────┬──────────────────────────────┬───────────────────────────┬──────────────────────────┬────────────────────────────┬───────────────────────────┬────────────────────────────────┬───────────────────────────────┬───────────────────────────────┬───────────────────────────────┬───────────────────────────────┬──────────────┬─────────────────────────────────────┬─────────────────────────────────────┬─────────────────────────────────────┬─────────────────────┬─────────────────────┐
│ CRASH DATE │ CRASH TIME │ BOROUGH  │ ZIP CODE │ LATITUDE  │ LONGITUDE │          LOCATION          │    ON STREET NAME    │  CROSS STREET NAME  │      OFF STREET NAME      │ NUMBER OF PERSONS INJURED │ NUMBER OF PERSONS KILLED │ NUMBER OF PEDESTRIANS INJURED

In [4]:
con.sql(f"SELECT * FROM read_csv('{DATA_B}') LIMIT 10")

┌────────────┬─────────────────────┬─────────────────────┬─────────┬─────────────────────────────────┬───────────────────────────────────┬──────────────────────────────────────┬────────────────────┬─────────────────┬──────────────┬─────────────────────────┬───────────────────┬────────────────┬─────────────────┬───────────────────────┬───────────────────────┬──────────────┬───────────────┬───────────────────┬───────────────┬─────────┬──────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────┬─────────────────┬──────────────────┬─────────────────┬────────────┬─────────┬────────────────────────────┬────────────────────────────┬────────────────────────┬────────────────────┬──────────────┬──────────────┬──────────────────────┬───────────────────────┬─────────────────────┬──────────────────────────┬───────────┬────────────────────────┬────────────────┬─────────────────┬─

## 4. Load both and check the types

Load each file into a table, then `DESCRIBE` it. Compare each column's type with what it *should* be. A number or date column stored as `VARCHAR` (text) is the same kind of problem you fixed in class.

In [5]:
for table, path in ((TABLE_A, DATA_A), (TABLE_B, DATA_B)):
    con.sql(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM read_csv('{path}')")
    rows = con.sql(f"SELECT count(*) FROM {table}").fetchone()[0]
    print(f"{table}: {rows:,} rows loaded")

crashes: 291,998 rows loaded
speeding: 2,697 rows loaded


In [6]:
con.sql(f"DESCRIBE {TABLE_A}")

┌───────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name          │ column_type │  null   │   key   │ default │  extra  │
│            varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ CRASH DATE                    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ CRASH TIME                    │ TIME        │ YES     │ NULL    │ NULL    │ NULL    │
│ BOROUGH                       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ ZIP CODE                      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ LATITUDE                      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ LONGITUDE                     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ LOCATION                      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ ON STREET NAME                

In [7]:
con.sql(f"DESCRIBE {TABLE_A}").df()
# Was not sure how to see the middle rows, so using a dataframe.

,column_name,column_type,null,key,default,extra
0,CRASH DATE,VARCHAR,YES,None,None,None
1,CRASH TIME,TIME,YES,None,None,None
2,BOROUGH,VARCHAR,YES,None,None,None
3,ZIP CODE,BIGINT,YES,None,None,None
4,LATITUDE,DOUBLE,YES,None,None,None
5,LONGITUDE,DOUBLE,YES,None,None,None
6,LOCATION,VARCHAR,YES,None,None,None
7,ON STREET NAME,VARCHAR,YES,None,None,None
8,CROSS STREET NAME,VARCHAR,YES,None,None,None
9,OFF STREET NAME,VARCHAR,YES,None,None,None


In [8]:
con.sql(f"DESCRIBE {TABLE_B}")

┌──────────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│             column_name              │ column_type │  null   │   key   │ default │  extra  │
│               varchar                │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ Unique Key                           │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Created Date                         │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ Closed Date                          │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ Agency                               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Agency Name                          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Problem (formerly Complaint Type)    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Problem Detail (formerly Descriptor) │ VARCHAR  

In [9]:
con.sql(f"DESCRIBE {TABLE_B}").df()

,column_name,column_type,null,key,default,extra
0,Unique Key,BIGINT,YES,None,None,None
1,Created Date,TIMESTAMP,YES,None,None,None
2,Closed Date,TIMESTAMP,YES,None,None,None
3,Agency,VARCHAR,YES,None,None,None
4,Agency Name,VARCHAR,YES,None,None,None
5,Problem (formerly Complaint Type),VARCHAR,YES,None,None,None
6,Problem Detail (formerly Descriptor),VARCHAR,YES,None,None,None
7,Additional Details,VARCHAR,YES,None,None,None
8,Location Type,VARCHAR,YES,None,None,None
9,Incident Zip,BIGINT,YES,None,None,None


`SUMMARIZE` profiles every column at once: min, max, how many distinct values, and `null_percentage` (how much is missing).

In [9]:
con.sql(f"SUMMARIZE {TABLE_A}")

┌───────────────────────────────┬─────────────┬──────────────────────────────┬────────────────────────────┬───────────────┬────────────────────┬───────────────────┬────────────────────┬───────────────────┬────────────────────┬────────┬─────────────────┐
│          column_name          │ column_type │             min              │            max             │ approx_unique │        avg         │        std        │        q25         │        q50        │        q75         │ count  │ null_percentage │
│            varchar            │   varchar   │           varchar            │          varchar           │     int64     │      varchar       │      varchar      │      varchar       │      varchar      │      varchar       │ int64  │  decimal(9,2)   │
├───────────────────────────────┼─────────────┼──────────────────────────────┼────────────────────────────┼───────────────┼────────────────────┼───────────────────┼────────────────────┼───────────────────┼────────────────────┼────────┼───

In [10]:
con.sql(f"SUMMARIZE {TABLE_B}")

┌──────────────────────────────────────┬─────────────┬─────────────────────────────────────────┬─────────────────────────────────────────┬───────────────┬────────────────────────────┬─────────────────────┬────────────────────────────┬────────────────────────────┬────────────────────────────┬───────┬─────────────────┐
│             column_name              │ column_type │                   min                   │                   max                   │ approx_unique │            avg             │         std         │            q25             │            q50             │            q75             │ count │ null_percentage │
│               varchar                │   varchar   │                 varchar                 │                 varchar                 │     int64     │          varchar           │       varchar       │          varchar           │          varchar           │          varchar           │ int64 │  decimal(9,2)   │
├──────────────────────────────────────┼───

## 5. Fix what loaded wrong (if anything)

If every type looks right, write "No fixes needed" in the text cell after the next one and skip ahead.

Otherwise, open the CSV (double-click it in the `data` folder) and find **how missing values are written** in the columns that came in as text. Edit the `nullstr` list below to match, and run it for the table that needs fixing.

In [12]:
# Change TABLE_A / DATA_A to TABLE_B / DATA_B if it's the second table that needs fixing
con.sql(f"""
    CREATE OR REPLACE TABLE {TABLE_A} AS
    SELECT * FROM read_csv('{DATA_A}', nullstr = [''])
""")
con.sql(f"DESCRIBE {TABLE_A}")

┌───────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name          │ column_type │  null   │   key   │ default │  extra  │
│            varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ CRASH DATE                    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ CRASH TIME                    │ TIME        │ YES     │ NULL    │ NULL    │ NULL    │
│ BOROUGH                       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ ZIP CODE                      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ LATITUDE                      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ LONGITUDE                     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ LOCATION                      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ ON STREET NAME                

In [ ]:
# Since the CRASH DATE remained as VARCHAR, I wanted to see if any date was missing.
con.sql("""
    SELECT count(*) - count("CRASH DATE")          AS missing_date,
    FROM crashes
""")

┌──────────────┐
│ missing_date │
│    int64     │
├──────────────┤
│            0 │
└──────────────┘

In [13]:
# Decided to change the data type of CRASH DATE.
con.sql(f"""
    CREATE OR REPLACE TABLE crashes_temp AS
    SELECT * REPLACE (
        STRPTIME("CRASH DATE", '%m/%d/%Y')::DATE AS "CRASH DATE"
    )
    FROM {TABLE_A}
""")

con.sql(f"DROP TABLE {TABLE_A}")
con.sql(f"ALTER TABLE crashes_temp RENAME TO {TABLE_A}")
con.sql(f"DESCRIBE {TABLE_A}")

┌───────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name          │ column_type │  null   │   key   │ default │  extra  │
│            varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ CRASH DATE                    │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ CRASH TIME                    │ TIME        │ YES     │ NULL    │ NULL    │ NULL    │
│ BOROUGH                       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ ZIP CODE                      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ LATITUDE                      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ LONGITUDE                     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ LOCATION                      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ ON STREET NAME                

**What did you fix, and how did you know?** (Double-click to edit.)
- I fixed the data type of the CRASH DATE column from VARCHAR to DATE. Looking at the csv, I saw blanks (" ") for empty values. When the nullstr command still returned VARCHAR, I checked to see if any dates were missing. Since no dates were missing, it was an issue of the data type itself and changing it.


## 6. Find the key: how do the two tables connect?

The **key** is the column both tables share. Type its name in each table below. It may be spelled differently in each, e.g. `Country Code` in one and `iso3` in the other.

The next cell counts the distinct key values in each table and how many of them appear in both. (It treats both as text, so a ZIP code stored as a number still matches one stored as text.)

In [14]:
KEY_A = "ZIP CODE"   # <- the shared column's name in your main table
KEY_B = "Incident Zip"   # <- its name in the second table

In [15]:
con.sql(f"""
    WITH a AS (SELECT DISTINCT CAST("{KEY_A}" AS VARCHAR) AS k FROM {TABLE_A}),
         b AS (SELECT DISTINCT CAST("{KEY_B}" AS VARCHAR) AS k FROM {TABLE_B})
    SELECT
        (SELECT count(*) FROM a)                           AS keys_in_a,
        (SELECT count(*) FROM b)                           AS keys_in_b,
        (SELECT count(*) FROM a WHERE k IN (SELECT k FROM b)) AS keys_in_both
""")

┌───────────┬───────────┬──────────────┐
│ keys_in_a │ keys_in_b │ keys_in_both │
│   int64   │   int64   │    int64     │
├───────────┼───────────┼──────────────┤
│       226 │       171 │          170 │
└───────────┴───────────┴──────────────┘

**Describe the relationship** (double-click to edit):

- **Key column:** in the main table it's called "ZIP CODE", in the second "Incident Zip".
- **How they relate:** does one row in the second table match *one* row in the main table, or *many*? (e.g. one country has many GDP rows, one per year)
- - Both tables have many rows that match since each row in the main table is a distinct car crash and each row in the second table is a distinct 311 report of a vehicle speeding. In both tables, the zip code is an identifier of location so it can repeat many times.
- **How well they match:** how many keys appear in only one table? Give one example and a guess at why (spelling, codes vs. names, regions mixed in with countries...)
- - The keys match pretty well considering 170 appear in both tables and Table B has 171 keys. Perhaps 1 zip code (in Table B, 311 reports) with reported speeding had no recorded vehicle crashes (in Table A, vehicle crashes). At the same time, 56 zip codes appear in Table A of vehicle crashes that are not in Table B. Crashes can happen for many reasons throughout the city and may not overlap with reported speeding.



## 7. Five queries that explain your data

Write one query of each kind. Under each, double-click the text cell and write one line: **what does this result tell us?** Use either table.

Tip: if a column name has spaces or capitals, wrap it in double quotes: `"Country Name"`. Start from a query in the in-class notebook and change one thing.

### Query 1 · Size and shape
*How many rows, and over what time period or area?* Useful SQL: `count(*), min(), max()`

In [16]:
con.sql("""
    SELECT
        count(*) AS row_count,
        min("CRASH DATE") AS earliest_crash,
        max("CRASH DATE") AS latest_crash
    FROM crashes
""")

┌───────────┬────────────────┬──────────────┐
│ row_count │ earliest_crash │ latest_crash │
│   int64   │      date      │     date     │
├───────────┼────────────────┼──────────────┤
│    291998 │ 2022-01-01     │ 2025-01-01   │
└───────────┴────────────────┴──────────────┘

**What this tells us:** 
- From January 1, 2022, to January 1, 2025, there were 291,998 vehicle crashes in NYC over three years.

### Query 2 · Filter
*Which rows meet a condition you care about?* Useful SQL: `WHERE`

In [17]:
con.sql("""
    SELECT
        count(*) AS fatal_crashes,
    FROM crashes
    WHERE "NUMBER OF PERSONS KILLED" > 0
""")

┌───────────────┐
│ fatal_crashes │
│     int64     │
├───────────────┤
│           801 │
└───────────────┘

**What this tells us:**
- - Of the almost 292,000 crashes recorded over the three year span, 801 were fatal.

### Query 3 · Group
*How does a count or average differ across categories?* Useful SQL: `GROUP BY`

In [22]:
con.sql("""
    SELECT "BOROUGH",
           count(*) AS speeding_reports,
    FROM speeding
    GROUP BY "BOROUGH"
""")

┌───────────────┬──────────────────┐
│    Borough    │ speeding_reports │
│    varchar    │      int64       │
├───────────────┼──────────────────┤
│ BRONX         │              114 │
│ STATEN ISLAND │              115 │
│ BROOKLYN      │              291 │
│ QUEENS        │             2026 │
│ MANHATTAN     │              151 │
└───────────────┴──────────────────┘

**What this tells us:** 
- Queens has the highest number of speeding complaints to 311. This does not mean it has the highest number of speeders or speeding tickets. The 10x number of reports compared to the other boroughs is worth investigating. Is there an area with exponentially more speeding, are people in Queens more likely to report, is there perhaps a single user overly reporting and skewing the data, or is this an accurate representation?

### Query 4 · Rank
*What are the top or bottom 10 of something?* Useful SQL: `ORDER BY ... LIMIT 10`

In [48]:
con.sql("""
    SELECT
        "ZIP CODE",
        COUNT(*) AS crashes,
        SUM("NUMBER OF PERSONS INJURED") AS persons_injured,
        SUM("NUMBER OF PERSONS KILLED") AS persons_killed
    FROM crashes
    GROUP BY "ZIP CODE"
    ORDER BY persons_injured DESC
    LIMIT 10
""")

┌──────────┬─────────┬─────────────────┬────────────────┐
│ ZIP CODE │ crashes │ persons_injured │ persons_killed │
│  int64   │  int64  │     int128      │     int128     │
├──────────┼─────────┼─────────────────┼────────────────┤
│     NULL │   92135 │           57181 │            380 │
│    11207 │    4946 │            2967 │              7 │
│    11236 │    3279 │            2202 │              9 │
│    11208 │    3259 │            1805 │              1 │
│    11203 │    2865 │            1803 │              1 │
│    11212 │    3102 │            1723 │              3 │
│    11226 │    2670 │            1722 │              7 │
│    11234 │    2779 │            1697 │              5 │
│    11385 │    2664 │            1314 │              2 │
│    11434 │    2100 │            1299 │              6 │
└──────────┴─────────┴─────────────────┴────────────────┘
  10 rows                                     4 columns

**What this tells us:** 
- There are 92,135 crashes without a reported ZIP CODE, which is almost a third of the crashes. Otherwise, 11207 has the most amount of people injured in crashes.

### Query 5 · Quality check
*What's missing, duplicated or impossible?* Useful SQL: `count(*) - count(column), count(DISTINCT ...)`

In [49]:

con.sql("""
    SELECT count(*) - count("CRASH DATE") AS missing_date,
        count(*) - count("CRASH TIME") AS missing_time,
        count(*) - count("BOROUGH") AS missing_borough,
        count(*) - count("ZIP CODE") AS missing_zip,
        count(*) - count("LATITUDE") AS missing_lat,
        count(*) - count("LONGITUDE") AS missing_long,
        count(*) - count("NUMBER OF PERSONS INJURED") AS missing_injuries,
        count(*) - count("NUMBER OF PERSONS KILLED") AS missing_deaths,
    FROM crashes
""")

┌──────────────┬──────────────┬─────────────────┬─────────────┬─────────────┬──────────────┬──────────────────┬────────────────┐
│ missing_date │ missing_time │ missing_borough │ missing_zip │ missing_lat │ missing_long │ missing_injuries │ missing_deaths │
│    int64     │    int64     │      int64      │    int64    │    int64    │    int64     │      int64       │     int64      │
├──────────────┼──────────────┼─────────────────┼─────────────┼─────────────┼──────────────┼──────────────────┼────────────────┤
│            0 │            0 │           92097 │       92135 │       23325 │        23325 │                0 │              0 │
└──────────────┴──────────────┴─────────────────┴─────────────┴─────────────┴──────────────┴──────────────────┴────────────────┘

In [50]:
con.sql("""
    SELECT count(*) - count("Created Date") AS missing_date,
        count(*) - count("Incident Zip") AS missing_zip,
        count(*) - count("Borough") AS missing_borough,
        count(*) - count("Vehicle Type") AS missing_vehicle,
        count(*) - count("Latitude") AS missing_lat,
        count(*) - count("Longitude") AS missing_long,
    FROM speeding
""")

┌──────────────┬─────────────┬─────────────────┬─────────────────┬─────────────┬──────────────┐
│ missing_date │ missing_zip │ missing_borough │ missing_vehicle │ missing_lat │ missing_long │
│    int64     │    int64    │      int64      │      int64      │    int64    │    int64     │
├──────────────┼─────────────┼─────────────────┼─────────────────┼─────────────┼──────────────┤
│            0 │           0 │               0 │            2697 │          45 │           45 │
└──────────────┴─────────────┴─────────────────┴─────────────────┴─────────────┴──────────────┘

**What this tells us:** 
- Table A is missing the borough for 92,097 crashes and missing zip codes for 92,135 crashes. Slighlty more zip codes are missing than boroughs. Of the recorded crashes, less are missing lat and long.
- Table B is not missing any zip codes and is more robust for joining with this key.

## 8. Three observations

Specific things you noticed that you'd want to check before trusting this data. Point to numbers.

1. Of the 801 fatal crashes in Table A, 380 crashes are missing a zip code. I would want to check for lat and long coodrinates for these crashes. With more steps, the coordinates could provide a better work around for geographic information that could lead to filling in missing zip codes if one chooses to not dismiss all of these data points.
2. Table B from the 311 reports seems more robust. I would want to check all the columns for missing data and see why 45 reports are missing the lat, long coordinates as this could indicate reasons for dismissing these data points.
3. The 2026 reports from Queens need to be checked for duplicate time stamps and duplicate information within a short window of time to determine repeat submissions for the same speeding incident.

**One question you'd want a non-technical person to be able to ask of this data:**

- How many people in NYC are injured annually in vehicle crashes where the most speeding is reported?

## 9. Dataset cards

| | Main table | Second table |
|---|---|---|
| **Dataset name** | Motor Vehicle Collisions - Crashes | 311 Service Requests from 2020 to Present |
| **Source (link)** | https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95/explore/query/SELECT%0A%20%20%60crash_date%60%2C%0A%20%20%60crash_time%60%2C%0A%20%20%60borough%60%2C%0A%20%20%60zip_code%60%2C%0A%20%20%60latitude%60%2C%0A%20%20%60longitude%60%2C%0A%20%20%60location%60%2C%0A%20%20%60on_street_name%60%2C%0A%20%20%60off_street_name%60%2C%0A%20%20%60cross_street_name%60%2C%0A%20%20%60number_of_persons_injured%60%2C%0A%20%20%60number_of_persons_killed%60%2C%0A%20%20%60number_of_pedestrians_injured%60%2C%0A%20%20%60number_of_pedestrians_killed%60%2C%0A%20%20%60number_of_cyclist_injured%60%2C%0A%20%20%60number_of_cyclist_killed%60%2C%0A%20%20%60number_of_motorist_injured%60%2C%0A%20%20%60number_of_motorist_killed%60%2C%0A%20%20%60contributing_factor_vehicle_1%60%2C%0A%20%20%60contributing_factor_vehicle_2%60%2C%0A%20%20%60contributing_factor_vehicle_3%60%2C%0A%20%20%60contributing_factor_vehicle_4%60%2C%0A%20%20%60contributing_factor_vehicle_5%60%2C%0A%20%20%60collision_id%60%2C%0A%20%20%60vehicle_type_code1%60%2C%0A%20%20%60vehicle_type_code2%60%2C%0A%20%20%60vehicle_type_code_3%60%2C%0A%20%20%60vehicle_type_code_4%60%2C%0A%20%20%60vehicle_type_code_5%60%0AWHERE%0A%20%20%60crash_date%60%0A%20%20%20%20BETWEEN%20%222022-01-01T00%3A00%3A00%22%20%3A%3A%20floating_timestamp%0A%20%20%20%20AND%20%222025-01-01T00%3A00%3A00%22%20%3A%3A%20floating_timestamp/page/filter | https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2020-to-Present/erm2-nwe9/explore/query/SELECT%0A%20%20%60unique_key%60%2C%0A%20%20%60created_date%60%2C%0A%20%20%60closed_date%60%2C%0A%20%20%60agency%60%2C%0A%20%20%60agency_name%60%2C%0A%20%20%60complaint_type%60%2C%0A%20%20%60descriptor%60%2C%0A%20%20%60descriptor_2%60%2C%0A%20%20%60location_type%60%2C%0A%20%20%60incident_zip%60%2C%0A%20%20%60incident_address%60%2C%0A%20%20%60street_name%60%2C%0A%20%20%60cross_street_1%60%2C%0A%20%20%60cross_street_2%60%2C%0A%20%20%60intersection_street_1%60%2C%0A%20%20%60intersection_street_2%60%2C%0A%20%20%60address_type%60%2C%0A%20%20%60city%60%2C%0A%20%20%60landmark%60%2C%0A%20%20%60facility_type%60%2C%0A%20%20%60status%60%2C%0A%20%20%60due_date%60%2C%0A%20%20%60resolution_description%60%2C%0A%20%20%60resolution_action_updated_date%60%2C%0A%20%20%60community_board%60%2C%0A%20%20%60council_district%60%2C%0A%20%20%60police_precinct%60%2C%0A%20%20%60bbl%60%2C%0A%20%20%60borough%60%2C%0A%20%20%60x_coordinate_state_plane%60%2C%0A%20%20%60y_coordinate_state_plane%60%2C%0A%20%20%60open_data_channel_type%60%2C%0A%20%20%60park_facility_name%60%2C%0A%20%20%60park_borough%60%2C%0A%20%20%60vehicle_type%60%2C%0A%20%20%60taxi_company_borough%60%2C%0A%20%20%60taxi_pick_up_location%60%2C%0A%20%20%60bridge_highway_name%60%2C%0A%20%20%60bridge_highway_direction%60%2C%0A%20%20%60road_ramp%60%2C%0A%20%20%60bridge_highway_segment%60%2C%0A%20%20%60latitude%60%2C%0A%20%20%60longitude%60%2C%0A%20%20%60location%60%0AWHERE%0A%20%20caseless_one_of%28%60agency%60%2C%20%22NYPD%22%29%0A%20%20AND%20caseless_one_of%28%60descriptor%60%2C%20%22Chronic%20Speeding%22%29%0A%20%20AND%20%60created_date%60%0A%20%20%20%20%20%20%20%20BETWEEN%20%222022-01-01T00%3A00%3A00%22%20%3A%3A%20floating_timestamp%0A%20%20%20%20%20%20%20%20AND%20%222025-01-01T00%3A00%3A00%22%20%3A%3A%20floating_timestamp%0AORDER%20BY%20%60created_date%60%20DESC%20NULL%20FIRST/page/filter |
| **Who collected it, and why** | NYPD | 311, under the Office of Technology and Innovation (OTI). |
| **License or terms of use** | Public data - New York City’s Open Data Law | Public data - New York City’s Open Data Law |
| **Rows × columns** | 291,998 rows x 29 columns | 2,697 rows x 44 columns |
| **Time period covered** | 3 years | 3 years |
| **How often it's updated** | Daily, but last updated June 15, 2026 | Daily |

## 10. Submit

1. Run the cell below to close the connection.
2. Fill in the lifecycle map of your data in `labs/lab1.md`.
3. **Save this notebook (Cmd+S)**, then commit and push: **Source Control** → message `Lab 1` → **Commit** → **Sync Changes**.
4. Check on GitHub that both CSVs, both notebooks and `labs/lab1.md` are there, with results visible.
5. Paste your repo URL into the Lab 1 assignment on CourseWorks.

In [52]:
con.close()
print("Done. Save, commit, push, and submit your repo URL.")

Done. Save, commit, push, and submit your repo URL.
